<div style="text-align: center;">
    <img src="./assets/ki_ag_main.jpg" alt="Logo" style="width: 1000px;"/>
</div>

<center><h1>KI-AG Workshop - Fine Tuning</h1></center>

<center>
    
</center>

In diesem Notebook erhältst du eine kurze Einführung in das Fine-Tuning von Sprachmodellen.



# Kapitel 1: Einrichten der Umgebung

📌 **Kurzanleitung: venv in VS Code einrichten & nutzen**

1️ **In die venv wechseln**
- In den Projektordner wechseln, z. B. `cd C:\Tools\2_fine_tuning`.
- Falls noch nicht vorhanden: ` py -3.12 -m venv .venv` ausführen.
- Die Umgebung aktivieren: `.\.venv\Scripts\Activate.ps1`. In der Eingabeaufforderung sollte jetzt `(.venv)` erscheinen.

2 **Abhängikeiten installieren**
- `pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121`
- `pip install transformers peft trl datasets accelerate sentencepiece protobuf bitsandbytes`

6️ **Tipp: Umgebung wieder verlassen**
- Nach getaner Arbeit `deactivate` eingeben.


# Kapitel 2: Fine Tuning mit `train.py`

In diesem Schritt lernst du, wie du mit dem Skript `scripts/train.py` und `scripts/chat_lora.py`sowohl ein LoRA-Fine-Tuning durchführst als auch anschließend direkt mit dem trainierten Adapter chattest.

## Was ist das Alpaca-Format?
- Das **Alpaca-Format** ist ein schlankes Supervised-Fine-Tuning-Format mit genau drei Feldern pro Beispiel:
  - `instruction` – beschreibt die generelle Aufgabe oder Rolle.
  - `input` – liefert den konkreten Kontext oder die Frage (kann auch leer sein).
  - `output` – enthält die gewünschte Antwort, die das Modell lernen soll.
- Gespeichert wird in einer Datei mit der Endung **`.jsonl` (JSON Lines)**. Jede Zeile ist ein eigenständiges JSON-Objekt, sodass große Datenmengen zeilenweise verarbeitet werden können.
- Wichtig: Keine zusätzlichen Kommata oder Zeilenumbrüche innerhalb des JSON-Objekts; jede Zeile muss valides JSON sein.

## Schritt-für-Schritt-Anleitung

## Aufgabe 1a:

Wir möchten zuerst dem Modell beibringen, einen bestimmten Namen, deinen eigenen, einzubrennen ohne ihm im Systemprompt anzugeben.

1️⃣ **Datensatz auswählen**
- Im Ordner `dataset/` liegen vorbereitete `.jsonl`-Dateien (z. B. `aufgabe1a.jsonl`). Jede Zeile folgt dem Alpaca-Format.
- Passe diese Zeile an, sodass er sich nennt wie du selbst (in `output`)

2️⃣ **Fine-Tuning starten**
- Aktiviere deine venv (siehe Kapitel 1) und stelle sicher, dass PyTorch installiert ist.
- Starte das Skript im Projektordner: `python scripts/train.py`
- Wähle im Menü „Training durchführen“ und anschließend:
  1. Den Datensatz aus `dataset/`
  2. Ein Basis-Modell aus `models/base`
  3. LoRA-Hyperparameter (Standardwerte funktionieren; siehe Hinweise unten)
- Bestätige, das Skript trainiert und speichert den Adapter automatisch.

3️⃣ **Trainergebnis testen**
- Nach dem Training fragt dich das Skript, ob du chatten möchtest – bestätige mit „j“.
- Stelle dem Modell passende Fragen (z. B. „Wie heißt du?“ bei `aufgabe1a.jsonl`) und prüfe die Reaktion.
- Variiere die Eingaben, um zu sehen, wie generalisiert das Verhalten wurde.

💡 **Hinweis:** Auch wenn der Datensatz schon vorbereitet ist, kannst du ihn in `dataset/` jederzeit erweitern. Jede zusätzliche Zeile (valide JSON-Zeile) verstärkt das gewünschte Verhalten.

🗒️ **Optionaler Bonus:** Lege im Ordner `systemprompts/` eine eigene `.txt`-Datei mit deinem Systemprompt an. Beim Chatten kannst du diesen Prompt auswählen und das Modell weiter steuern.

## Hinweise zu wichtigen Parametern
- **LoRA-Rank (r):** Mehr Kapazität (z. B. 8–16) lässt das Modell stärker umlernen. Für unser Minimalbeispiel empfehlen wir **r=16**, damit der Effekt deutlich sichtbar wird.
- **Alpha:** Gewichtet die LoRA-Aktualisierung. Setze sie bewusst hoch, z. B. **alpha=32**, um den gewünschten Output durchzudrücken – Overfitting ist hier ausdrücklich erwünscht.
- **Dropout:** Für maximalen Effekt **Dropout auf 0** setzen. So wird nichts «verwässert» und die wenigen Beispiele prägen sich stärker ein.
- **Trainingsläufe / Epochen:** Du kannst mehrere Runs hintereinander starten oder im Code `num_train_epochs` erhöhen (z. B. auf 5). Mehr Wiederholungen = stärkeres Einprägen.
- **Temperatur & Top-p (Chat):** Temperatur niedrig halten (0.3–0.5) und Top-p auf 0.8–0.9, damit die gelernte Antwort deterministischer zurückkommt.
- **Systemprompt:** Für das Testen des Namens hilft ein passender Prompt in `systemprompts/name.txt`, damit der Chat denselben Kontext bekommt wie im Training.
